# Music Annotation Project V8.2b
## Spatial Probability-Band, Route-Context and Confidence-Localisation Audit

**Runtime:** CPU is sufficient; GPU is not required.

V8.2b is a complete, executable development-only diagnostic derived from the V8.2a codebase. It does **not** retrain, refit, recalibrate or alter the V8.2 CNNs or isotonic calibrators. It reads the saved source-group OOF probabilities from one completed V8.2 run, copies the V8.2 files of record into its own run directory, and localises the probability mass that V8.2a exposed between 0.80 and 0.995.

### Primary questions
1. Where do the `0.80–0.95` PRINT-probability pixels occur on each page?
2. Which bands correspond to PRINT truth, ANNOTATION truth and truth-UNCERTAIN?
3. Are the pixels continuous structures or isolated/boundary fragments?
4. How do V7.8 route context, horizontal/vertical/compact support and mask depth vary by band?
5. What changes at `0.90` relative to `0.95`, at `0.85` relative to `0.90`, and at `0.80` relative to `0.85`?

### Diagnostic policies
- **Strict comparator:** final PRINT at `p >= 0.995`, with run-size filtering.
- **Exploratory automatic candidate:** final PRINT at `p >= 0.90`, with run-size filtering.
- **Tiered review:** automatic candidate at `p >= 0.90`; review-only tier at `0.80 <= p < 0.90`; high-confidence annotation remains separate.

Only the automatic candidate may change a generated corrected page, and even then V8.2b remains research evidence. The review tier is visual/audit evidence only.

### Amendment: extreme classifier-opinion experiment
This version adds read-only diagnostic renderings that collapse calibrated PRINT probability into WHITE / GREY / BLACK opinion states while retaining the five-state colour view. The synthetic opinion pages are not corrected pages, do not alter V7.8/V8.2 outputs, and have no promotion authority. They are scored post-hoc against existing development truth to expose where the aggressive black/white visual simplification succeeds or fails.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib

## 1. Configuration

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
V82_ROOT = ROOT / 'observed_print_classifier_v8_2'
OUTPUT_ROOT = ROOT / 'observed_print_classifier_v8_2b'

# Optional exact source. Leave None to use the most recently modified complete V8.2 run.
SOURCE_V82_RUN_OVERRIDE = None

# Diagnostic bands are half-open [low, high), except the first band includes 1.0.
BANDS = [
    {'band_id':'A_GE_0995',   'low':0.995, 'high':1.0000001, 'label':'>= 0.995',     'bgr':(0,220,0)},
    {'band_id':'B_095_0995',  'low':0.950, 'high':0.995,     'label':'0.950–0.995', 'bgr':(0,255,170)},
    {'band_id':'C_090_095',   'low':0.900, 'high':0.950,     'label':'0.900–0.950', 'bgr':(255,255,0)},
    {'band_id':'D_085_090',   'low':0.850, 'high':0.900,     'label':'0.850–0.900', 'bgr':(255,120,0)},
    {'band_id':'E_080_085',   'low':0.800, 'high':0.850,     'label':'0.800–0.850', 'bgr':(180,0,255)},
    {'band_id':'F_050_080',   'low':0.500, 'high':0.800,     'label':'0.500–0.800', 'bgr':(255,0,180)},
    {'band_id':'G_LT_050',    'low':0.000, 'high':0.500,     'label':'< 0.500',     'bgr':(110,110,110)},
]
STRICT_THRESHOLD = 0.995
AUTO_THRESHOLD = 0.900
REVIEW_THRESHOLD = 0.800
ANNOTATION_THRESHOLD = 0.995
ENABLE_RUN_FILTER = True
MIN_PRINT_RUN = 2
OVERLAY_ALPHA = 0.55
REVIEW_RADIUS = 1
HEATMAP_MIN_PROBABILITY = 0.50

# ============================================================
# TWEAKABLE DIAGNOSTIC VISUALISATION / OPINION EXPERIMENT
# Diagnostic only: changing these does NOT change V8.2b corrected pages.
# OpenCV colours are BGR rather than RGB.
# ============================================================
FIVE_STATE_DEFINITE_ANNOTATION_MAX = 0.10
FIVE_STATE_PROBABLE_ANNOTATION_MAX = 0.35
FIVE_STATE_UNSURE_MAX = 0.65
FIVE_STATE_PROBABLE_PRINT_MAX = 0.90

COLOUR_DEFINITELY_ANNOTATION = (0, 0, 210)    # dark red
COLOUR_PROBABLY_ANNOTATION   = (0, 128, 255)  # orange
COLOUR_UNSURE                = (220, 90, 20)  # blue
COLOUR_PROBABLY_PRINT        = (60, 255, 120) # lime
COLOUR_DEFINITELY_PRINT      = (0, 160, 0)    # green

FIVE_STATE_OVERLAY_ALPHA = 0.62
FIVE_STATE_REVIEW_RADIUS = 1
SHOW_RANGES_IN_LEGEND = True

# Three-level classifier opinion requested for the extreme diagnostic:
# p_print < 0.35 -> WHITE; 0.35 <= p_print < 0.65 -> GREY; p_print >= 0.65 -> BLACK.
OPINION_WHITE_BELOW = FIVE_STATE_PROBABLE_ANNOTATION_MAX
OPINION_BLACK_FROM = FIVE_STATE_UNSURE_MAX
OPINION_GREY_VALUE = 160
OPINION_APPLY_CONNECTED_RUN_FILTER = False
OPINION_MIN_BLACK_RUN = 2

CONTACT_SHEET_PAGES = 16
COPY_COMPLETE_CORE_RECORD = True

RUN = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_2b_spatial_probability_band_audit'
OUT = OUTPUT_ROOT / RUN
RECORD = OUT / 'source_v82_record'
TABLES = OUT / 'tables'
WHOLE = OUT / 'whole_pages'
VISUAL = OUT / 'visual_audit'
AUDIT = OUT / 'audit'
for p in [OUT, RECORD, TABLES, WHOLE, VISUAL, AUDIT]:
    p.mkdir(parents=True, exist_ok=False)

if not (0.5 <= REVIEW_THRESHOLD < AUTO_THRESHOLD < STRICT_THRESHOLD <= 1.0):
    raise ValueError('Require 0.5 <= REVIEW_THRESHOLD < AUTO_THRESHOLD < STRICT_THRESHOLD <= 1.0')
visual_edges=[0.0,FIVE_STATE_DEFINITE_ANNOTATION_MAX,FIVE_STATE_PROBABLE_ANNOTATION_MAX,FIVE_STATE_UNSURE_MAX,FIVE_STATE_PROBABLE_PRINT_MAX,1.0]
if visual_edges != sorted(visual_edges) or len(set(visual_edges)) != len(visual_edges):
    raise ValueError(f'Five-state visual boundaries must be strictly ordered: {visual_edges}')
if not (0.0 <= OPINION_WHITE_BELOW < OPINION_BLACK_FROM <= 1.0):
    raise ValueError('Require 0 <= OPINION_WHITE_BELOW < OPINION_BLACK_FROM <= 1.')
if not (0 <= OPINION_GREY_VALUE <= 255):
    raise ValueError('OPINION_GREY_VALUE must be in [0,255].')
print('V8.2b output:', OUT)

## 2. Resolve and validate one completed V8.2 source run

In [ ]:
REQUIRED_SOURCE_FILES = [
    'experiment_identity.json',
    'development_summary.json',
    'executive_summary.txt',
    'tables/all_eligible_oof_predictions.parquet',
    'audit/model_inventory.csv',
    'audit/calibrator_inventory.csv',
    'inputs/fold_map.csv',
    'inputs/channel_schema.json',
]

def complete_runs(root):
    hits=[]
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in REQUIRED_SOURCE_FILES):
                hits.append(p)
    return sorted(hits, key=lambda p:p.stat().st_mtime)

if SOURCE_V82_RUN_OVERRIDE is not None:
    SOURCE=Path(SOURCE_V82_RUN_OVERRIDE)
    missing=[r for r in REQUIRED_SOURCE_FILES if not (SOURCE/r).exists()]
    if missing: raise RuntimeError(f'Incomplete source override: {missing}')
else:
    hits=complete_runs(V82_ROOT)
    if not hits: raise RuntimeError(f'No complete V8.2 run found below {V82_ROOT}')
    SOURCE=hits[-1]

source_identity=json.loads((SOURCE/'experiment_identity.json').read_text())
if source_identity.get('version')!='8.2-development-whole-page':
    raise RuntimeError(f'Unexpected source version: {source_identity.get("version")}')
if source_identity.get('independent_data_used') is not False:
    raise RuntimeError('V8.2b may use only the development-only V8.2 OOF run.')

pred_path=SOURCE/'tables'/'all_eligible_oof_predictions.parquet'
full=pd.read_parquet(pred_path)
required={'page','source_group','y','x','truth','fold','raw_print','p_print'}
missing=required-set(full.columns)
if missing: raise RuntimeError(f'Prediction table missing columns: {sorted(missing)}')
for c in ['y','x','truth','fold']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(int)
for c in ['raw_print','p_print']:
    full[c]=pd.to_numeric(full[c],errors='raise').astype(np.float64)
if not np.isfinite(full[['raw_print','p_print']].to_numpy()).all():
    raise RuntimeError('Non-finite scores in source predictions.')
if full.duplicated(['page','y','x']).any():
    raise RuntimeError('Duplicate page/y/x rows in source predictions.')
full=full.reset_index(drop=True)
print('Source:',SOURCE)
print('Rows:',len(full),'Pages:',full.page.nunique(),'Groups:',full.source_group.nunique())

## 3. Copy and hash the complete V8.2 core record

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):
            h.update(b)
    return h.hexdigest()

def copy_item(src,dst):
    if src.is_dir(): shutil.copytree(src,dst)
    else:
        dst.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(src,dst)

if COPY_COMPLETE_CORE_RECORD:
    for d in ['inputs','models','calibrators','audit','tables']:
        src=SOURCE/d
        if not src.exists(): raise RuntimeError(f'Missing source record directory: {src}')
        copy_item(src,RECORD/d)
    for name in ['experiment_identity.json','development_summary.json','executive_summary.txt','software_environment.json']:
        src=SOURCE/name
        if src.exists(): copy_item(src,RECORD/name)

for item in source_identity.get('models',[]):
    copied=RECORD/'models'/Path(item['checkpoint']).name
    if not copied.exists() or sha256_file(copied)!=item['checkpoint_sha256']:
        raise RuntimeError(f'Copied model hash mismatch: {copied.name}')
for item in source_identity.get('calibrators',[]):
    copied=RECORD/'calibrators'/Path(item['joblib']).name
    if not copied.exists() or sha256_file(copied)!=item['joblib_sha256']:
        raise RuntimeError(f'Copied calibrator hash mismatch: {copied.name}')

inventory=[]
for p in sorted(RECORD.rglob('*')):
    if p.is_file(): inventory.append({'relative_path':str(p.relative_to(RECORD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inventory).to_csv(AUDIT/'copied_source_v82_artifact_inventory.csv',index=False)
source_record={'source_run':str(SOURCE),'source_identity_sha256':sha256_file(SOURCE/'experiment_identity.json'),'source_predictions_sha256':sha256_file(pred_path),'copied_predictions_sha256':sha256_file(RECORD/'tables'/'all_eligible_oof_predictions.parquet'),'copied_files':len(inventory),'copied_bytes':int(sum(x['bytes'] for x in inventory)),'model_hashes_verified':True,'calibrator_hashes_verified':True}
(OUT/'source_v82_record.json').write_text(json.dumps(source_record,indent=2))
print(json.dumps(source_record,indent=2))

## 4. Load V7.5/V7.8 route and support context from the exact V8.2 reference

In [ ]:
v75=Path(source_identity['v7_5_reference'])
status=pd.read_csv(v75/'tables'/'cache_status.csv')
cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
required_arrays=['mask','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
page_context={}
for page in sorted(full.page.unique()):
    if page not in cache: raise RuntimeError(f'No V7.5 cache for {page}')
    with np.load(cache[page]) as z:
        miss=[k for k in required_arrays if k not in z.files]
        if miss: raise RuntimeError(f'{page}: missing contextual arrays {miss}')
        ctx={k:z[k].copy() for k in required_arrays}
    mask=ctx['mask']>0
    n,labels,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
    ctx['mask_bool']=mask
    ctx['component_id']=labels.astype(np.int32)
    ctx['mask_depth']=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5).astype(np.float32)
    page_context[page]=ctx
print('Loaded context for',len(page_context),'pages')

## 5. Enrich every OOF row with band, route, support, mask depth and component identity

In [ ]:
def assign_band(p):
    for b in BANDS:
        if b['low'] <= p < b['high']: return b['band_id']
    return 'UNASSIGNED'

def route_name(ctx,y,x):
    if ctx['g80_preserve'][y,x]>0:return 'preserve'
    if ctx['g80_remove'][y,x]>0:return 'remove'
    if ctx['g80_ambiguous'][y,x]>0:return 'ambiguous'
    return 'none'

enriched=[]
for page,g in full.groupby('page',sort=False):
    ctx=page_context[page]
    q=g.copy(); ys=q.y.to_numpy(int); xs=q.x.to_numpy(int)
    h,w=ctx['mask_bool'].shape
    if (ys<0).any() or (xs<0).any() or (ys>=h).any() or (xs>=w).any(): raise RuntimeError(f'{page}: coordinate outside contextual arrays')
    q['band_id']=[assign_band(v) for v in q.p_print.to_numpy()]
    q['route']=[route_name(ctx,y,x) for y,x in zip(ys,xs)]
    q['horizontal_support']=(ctx['g80_horizontal'][ys,xs]>0)
    q['vertical_support']=(ctx['g80_vertical'][ys,xs]>0)
    q['compact_support']=(ctx['g80_compact'][ys,xs]>0)
    q['any_support']=q[['horizontal_support','vertical_support','compact_support']].any(axis=1)
    q['support_count']=q[['horizontal_support','vertical_support','compact_support']].sum(axis=1).astype(int)
    q['mask_depth']=ctx['mask_depth'][ys,xs]
    q['component_id']=ctx['component_id'][ys,xs]
    enriched.append(q)
enriched=pd.concat(enriched,ignore_index=True)
if (enriched.band_id=='UNASSIGNED').any(): raise RuntimeError('Unassigned probability band detected.')
enriched.to_parquet(TABLES/'band_enriched_oof_predictions.parquet',index=False)
print(enriched.band_id.value_counts().sort_index())

## 6. Connected-run identity at strict, automatic and review thresholds

In [ ]:
def connected_run_features(frame,threshold,column_prefix):
    run_size=np.zeros(len(frame),np.int32)
    run_id=np.zeros(len(frame),np.int32)
    survives=np.zeros(len(frame),bool)
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy(); ctx=page_context[page]; h,w=ctx['mask_bool'].shape
        candidate=g.p_print.to_numpy()>=threshold
        pm=np.zeros((h,w),np.uint8)
        ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
        pm[ys[candidate],xs[candidate]]=1
        n,lab,stats,_=cv2.connectedComponentsWithStats(pm,8)
        labels=lab[ys,xs]
        sizes=np.zeros_like(labels)
        valid=labels>0
        sizes[valid]=stats[labels[valid],cv2.CC_STAT_AREA]
        run_size[idx]=sizes; run_id[idx]=labels; survives[idx]=candidate & (sizes>=MIN_PRINT_RUN)
    frame[f'{column_prefix}_run_size']=run_size
    frame[f'{column_prefix}_run_id']=run_id
    frame[f'{column_prefix}_survives']=survives
    return frame

enriched=connected_run_features(enriched,STRICT_THRESHOLD,'strict')
enriched=connected_run_features(enriched,0.950,'p095')
enriched=connected_run_features(enriched,AUTO_THRESHOLD,'auto')
enriched=connected_run_features(enriched,0.850,'p085')
enriched=connected_run_features(enriched,REVIEW_THRESHOLD,'review')
enriched.to_parquet(TABLES/'band_enriched_oof_predictions.parquet',index=False)
print('Strict surviving:',int(enriched.strict_survives.sum()))
print('Auto surviving:',int(enriched.auto_survives.sum()))
print('Review-or-higher surviving:',int(enriched.review_survives.sum()))

## 7. Band summaries: pooled, page, source group, route, support and component

In [ ]:
truth_names={0:'ANNOTATION',1:'PRINT',2:'UNCERTAIN'}
enriched['truth_name']=enriched.truth.map(truth_names)

def opinion_state(p):
    if p < OPINION_WHITE_BELOW: return 'WHITE_annotation_lean'
    if p < OPINION_BLACK_FROM: return 'GREY_review'
    return 'BLACK_print_lean'

enriched['opinion_state']=enriched.p_print.map(opinion_state)

opinion_pooled=(enriched.groupby(['opinion_state','truth_name'],as_index=False).agg(
    pixels=('p_print','size'),mean_p_print=('p_print','mean'),pages=('page','nunique'),source_groups=('source_group','nunique')))
opinion_pooled.to_csv(TABLES/'opinion_experiment_truth_summary_pooled.csv',index=False)

opinion_by_page=(enriched.groupby(['page','source_group','opinion_state','truth_name'],as_index=False).agg(
    pixels=('p_print','size'),mean_p_print=('p_print','mean')))
opinion_by_page.to_csv(TABLES/'opinion_experiment_truth_summary_by_page.csv',index=False)

opinion_by_group=(enriched.groupby(['source_group','opinion_state','truth_name'],as_index=False).agg(
    pixels=('p_print','size'),mean_p_print=('p_print','mean'),pages=('page','nunique')))
opinion_by_group.to_csv(TABLES/'opinion_experiment_truth_summary_by_source_group.csv',index=False)

black_q=enriched[enriched.opinion_state=='BLACK_print_lean']
black_summary={
    'white_below':OPINION_WHITE_BELOW,'black_from':OPINION_BLACK_FROM,
    'black_pixels':int(len(black_q)),'black_true_print':int((black_q.truth==1).sum()),
    'black_false_annotation':int((black_q.truth==0).sum()),
    'black_truth_uncertain':int((black_q.truth==2).sum()),
    'black_pages':int(black_q.page.nunique()),'black_source_groups':int(black_q.source_group.nunique()),
    'diagnostic_only':True,'changes_corrected_pages':False,
}
(TABLES/'opinion_experiment_summary.json').write_text(json.dumps(black_summary,indent=2))

pooled=(enriched.groupby(['band_id','truth_name'],as_index=False).size().rename(columns={'size':'pixels'}))
pooled_total=enriched.groupby('band_id',as_index=False).size().rename(columns={'size':'band_pixels'})
pooled=pooled.merge(pooled_total,on='band_id'); pooled['fraction_of_band']=pooled.pixels/pooled.band_pixels
pooled.to_csv(TABLES/'band_summary_pooled.csv',index=False)

by_page=(enriched.groupby(['page','source_group','band_id','truth_name'],as_index=False).size().rename(columns={'size':'pixels'}))
by_page.to_csv(TABLES/'band_summary_by_page.csv',index=False)
by_group=(enriched.groupby(['source_group','band_id','truth_name'],as_index=False).size().rename(columns={'size':'pixels'}))
by_group.to_csv(TABLES/'band_summary_by_source_group.csv',index=False)

route=(enriched.groupby(['band_id','route','truth_name'],as_index=False).agg(pixels=('p_print','size'),mean_p_print=('p_print','mean'),mean_mask_depth=('mask_depth','mean'),any_support_rate=('any_support','mean'),horizontal_rate=('horizontal_support','mean'),vertical_rate=('vertical_support','mean'),compact_rate=('compact_support','mean')))
route.to_csv(TABLES/'route_context_band_analysis.csv',index=False)

support=(enriched.groupby(['band_id','truth_name','support_count'],as_index=False).agg(pixels=('p_print','size'),mean_p_print=('p_print','mean'),mean_mask_depth=('mask_depth','mean')))
support.to_csv(TABLES/'support_band_analysis.csv',index=False)

comp=(enriched[enriched.component_id>0].groupby(['page','source_group','component_id','band_id'],as_index=False).agg(pixels=('p_print','size'),true_print=('truth',lambda x:int((x==1).sum())),false_annotation=('truth',lambda x:int((x==0).sum())),truth_uncertain=('truth',lambda x:int((x==2).sum())),mean_probability=('p_print','mean'),max_probability=('p_print','max'),mean_mask_depth=('mask_depth','mean'),any_support_rate=('any_support','mean'),max_auto_run=('auto_run_size','max')))
comp['false_true_ratio']=comp.false_annotation/np.maximum(comp.true_print,1)
comp.to_csv(TABLES/'component_band_analysis.csv',index=False)

print(pooled.to_string(index=False))

## 8. Incremental threshold populations and diagnostic policies

In [ ]:
def policy_summary(mask,name):
    q=enriched[mask]
    return {'policy':name,'selected':len(q),'true_print':int((q.truth==1).sum()),'false_annotation':int((q.truth==0).sum()),'truth_uncertain':int((q.truth==2).sum()),'pages':int(q.page.nunique()),'source_groups':int(q.source_group.nunique()),'utility':int((q.truth==1).sum()-5*(q.truth==0).sum())}

strict=enriched.strict_survives
auto=enriched.auto_survives
tier_review=enriched.review_survives & ~enriched.auto_survives
increment_090_095=enriched.auto_survives & ~enriched.p095_survives
increment_085_090=enriched.p085_survives & ~enriched.auto_survives
increment_080_085=enriched.review_survives & ~enriched.p085_survives

policy_rows=[policy_summary(strict,'STRICT_GE_0995_RUN2'),policy_summary(auto,'AUTO_GE_090_RUN2'),policy_summary(tier_review,'REVIEW_080_TO_090_RUN2'),policy_summary(increment_090_095,'NEW_090_TO_095'),policy_summary(increment_085_090,'NEW_085_TO_090'),policy_summary(increment_080_085,'NEW_080_TO_085')]
pd.DataFrame(policy_rows).to_csv(TABLES/'diagnostic_policy_summary.csv',index=False)

increment_rows=[]
for name,mask in [('NEW_090_TO_095',increment_090_095),('NEW_085_TO_090',increment_085_090),('NEW_080_TO_085',increment_080_085)]:
    q=enriched[mask]
    z=q.groupby(['page','source_group','truth_name'],as_index=False).size().rename(columns={'size':'pixels'}); z['increment']=name; increment_rows.append(z)
pd.concat(increment_rows,ignore_index=True).to_csv(TABLES/'incremental_threshold_by_page.csv',index=False)
print(pd.DataFrame(policy_rows).to_string(index=False))

## 9. Probability heatmaps, band overlays, truth diagnostics and incremental overlays

In [ ]:
BAND_BY_ID={b['band_id']:b for b in BANDS}
TRUTH_BGR={0:(0,0,255),1:(0,220,0),2:(0,191,255)}

FIVE_STATE_DEFINITIONS=[
    ('Definitely annotation',0.0,FIVE_STATE_DEFINITE_ANNOTATION_MAX,COLOUR_DEFINITELY_ANNOTATION),
    ('Probably annotation',FIVE_STATE_DEFINITE_ANNOTATION_MAX,FIVE_STATE_PROBABLE_ANNOTATION_MAX,COLOUR_PROBABLY_ANNOTATION),
    ('Unsure',FIVE_STATE_PROBABLE_ANNOTATION_MAX,FIVE_STATE_UNSURE_MAX,COLOUR_UNSURE),
    ('Probably print',FIVE_STATE_UNSURE_MAX,FIVE_STATE_PROBABLE_PRINT_MAX,COLOUR_PROBABLY_PRINT),
    ('Definitely print',FIVE_STATE_PROBABLE_PRINT_MAX,1.0000001,COLOUR_DEFINITELY_PRINT),
]

def dilate(mask,radius=REVIEW_RADIUS):
    if radius<=0:return mask
    k=np.ones((2*radius+1,2*radius+1),np.uint8)
    return cv2.dilate(mask.astype(np.uint8),k)>0

def blend_masks(gray,masks_and_colours,alpha=OVERLAY_ALPHA):
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); colour=base.copy(); active=np.zeros(gray.shape,bool)
    for mask,bgr in masks_and_colours:
        colour[mask]=bgr; active|=mask
    out=base.copy(); out[active]=cv2.addWeighted(base,1-alpha,colour,alpha,0)[active]
    return out

def legend(image,title,items):
    if image.ndim==2: image = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    h,w=image.shape[:2]; scale=max(.42,min(.8,w/1800)); item_scale=scale*.68
    rows=[]; current=[]; used=12
    for label,bgr in items:
        width=max(155,int(len(label)*7*item_scale/0.5+52))
        if current and used+width>w-12: rows.append(current); current=[]; used=12
        current.append((label,bgr,width)); used+=width
    if current: rows.append(current)
    row_h=max(28,int(34*scale/0.6)); title_h=max(34,int(42*scale/0.6)); banner=title_h+row_h*len(rows)+8
    canvas=np.full((h+banner,w,3),255,np.uint8); canvas[banner:]=image
    cv2.putText(canvas,title,(12,int(title_h*.70)),cv2.FONT_HERSHEY_SIMPLEX,scale,(30,30,30),1,cv2.LINE_AA)
    for row_no,row in enumerate(rows):
        x=12; y=title_h+row_h*row_no+int(row_h*.65)
        for label,bgr,width in row:
            cv2.rectangle(canvas,(x,y-14),(x+18,y+4),bgr,-1)
            cv2.putText(canvas,label,(x+24,y+3),cv2.FONT_HERSHEY_SIMPLEX,item_scale,(30,30,30),1,cv2.LINE_AA); x+=width
    return canvas

def heatmap_overlay(gray,prob,valid):
    norm=np.clip((prob-HEATMAP_MIN_PROBABILITY)/(1-HEATMAP_MIN_PROBABILITY),0,1)
    heat=cv2.applyColorMap((norm*255).astype(np.uint8),cv2.COLORMAP_TURBO)
    base=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR); out=base.copy(); active=valid & (prob>=HEATMAP_MIN_PROBABILITY)
    out[active]=cv2.addWeighted(base,1-OVERLAY_ALPHA,heat,OVERLAY_ALPHA,0)[active]
    return out

def range_label(label,lo,hi):
    if not SHOW_RANGES_IN_LEGEND:return label
    close=']' if hi>1 else ')'
    return f'{label}  [{lo:.3f}–{min(hi,1.0):.3f}{close}'

def five_state_overlay(gray,prob,valid):
    layers=[]; items=[]
    for label,lo,hi,bgr in FIVE_STATE_DEFINITIONS:
        m=valid & (prob>=lo) & (prob<hi)
        layers.append((dilate(m,FIVE_STATE_REVIEW_RADIUS),bgr)); items.append((range_label(label,lo,hi),bgr))
    return blend_masks(gray,layers,alpha=FIVE_STATE_OVERLAY_ALPHA),items

def opinion_masks(prob,valid):
    white=valid & (prob<OPINION_WHITE_BELOW)
    grey=valid & (prob>=OPINION_WHITE_BELOW) & (prob<OPINION_BLACK_FROM)
    black=valid & (prob>=OPINION_BLACK_FROM)
    if OPINION_APPLY_CONNECTED_RUN_FILTER:
        n,lab,stats,_=cv2.connectedComponentsWithStats(black.astype(np.uint8),8)
        keep=np.zeros_like(black)
        for cid in range(1,n):
            if stats[cid,cv2.CC_STAT_AREA]>=OPINION_MIN_BLACK_RUN: keep|=(lab==cid)
        grey|=(black & ~keep); black=keep
    return white,grey,black

def binary_opinion_canvas(shape,white,grey,black):
    out=np.full(shape,255,np.uint8); out[grey]=OPINION_GREY_VALUE; out[black]=0
    return out

def binary_opinion_on_page(gray,white,grey,black):
    out=gray.copy(); out[white]=255; out[grey]=OPINION_GREY_VALUE; out[black]=0
    return out

def extreme_review_preview(gray,prob,valid):
    # Definite annotation -> white; both print-facing bands -> black; orange/blue remain visible for review.
    out=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR)
    definite_annotation=valid & (prob<FIVE_STATE_DEFINITE_ANNOTATION_MAX)
    orange=valid & (prob>=FIVE_STATE_DEFINITE_ANNOTATION_MAX) & (prob<FIVE_STATE_PROBABLE_ANNOTATION_MAX)
    blue=valid & (prob>=FIVE_STATE_PROBABLE_ANNOTATION_MAX) & (prob<FIVE_STATE_UNSURE_MAX)
    print_facing=valid & (prob>=FIVE_STATE_UNSURE_MAX)
    out[definite_annotation]=(255,255,255); out[orange]=COLOUR_PROBABLY_ANNOTATION
    out[blue]=COLOUR_UNSURE; out[print_facing]=(0,0,0)
    return out

page_inventory=[]
for page,g in enriched.groupby('page',sort=False):
    src=SOURCE/'whole_pages'/page
    observed=cv2.imread(str(src/'observed.png'),0); v78=cv2.imread(str(src/'v7_8.png'),0); v82=cv2.imread(str(src/'v8_2.png'),0)
    if observed is None or v78 is None or v82 is None or not (observed.shape==v78.shape==v82.shape): raise RuntimeError(f'{page}: missing/mismatched source whole pages')
    h,w=observed.shape; ys=g.y.to_numpy(int); xs=g.x.to_numpy(int)
    prob=np.zeros((h,w),np.float32); valid=np.zeros((h,w),bool); prob[ys,xs]=g.p_print.to_numpy(np.float32); valid[ys,xs]=True

    band_masks=[]; band_items=[]
    for b in BANDS[:-1]:
        m=np.zeros((h,w),bool); hit=(g.band_id==b['band_id']).to_numpy(); m[ys[hit],xs[hit]]=True
        band_masks.append((dilate(m),b['bgr'])); band_items.append((b['label'],b['bgr']))
    band_overlay=legend(blend_masks(observed,band_masks),f'{page} — V8.2b PRINT probability bands',band_items)
    heat=legend(heatmap_overlay(observed,prob,valid),f'{page} — V8.2b calibrated PRINT probability heatmap',[('0.50',(128,0,0)),('0.75',(255,255,0)),('1.00',(0,0,128))])

    five_img,five_items=five_state_overlay(observed,prob,valid)
    five=legend(five_img,f'{page} — V8.2b five-state annotation-to-print confidence',five_items)

    white,grey,black=opinion_masks(prob,valid)
    opinion=legend(binary_opinion_canvas(observed.shape,white,grey,black),f'{page} — DIAGNOSTIC classifier opinion only',[
        (f'WHITE p < {OPINION_WHITE_BELOW:.2f}',(255,255,255)),
        (f'GREY {OPINION_WHITE_BELOW:.2f} <= p < {OPINION_BLACK_FROM:.2f}',(OPINION_GREY_VALUE,)*3),
        (f'BLACK p >= {OPINION_BLACK_FROM:.2f}',(0,0,0))])
    opinion_page=legend(binary_opinion_on_page(observed,white,grey,black),f'{page} — DIAGNOSTIC classifier opinion on observed context',[
        ('WHITE annotation-facing',(255,255,255)),('GREY review',(OPINION_GREY_VALUE,)*3),('BLACK print-facing',(0,0,0))])
    extreme=legend(extreme_review_preview(observed,prob,valid),f'{page} — EXTREME state preview: white/black decisions, orange/blue review',[
        (f'WHITE p < {FIVE_STATE_DEFINITE_ANNOTATION_MAX:.2f}',(255,255,255)),
        (f'ORANGE {FIVE_STATE_DEFINITE_ANNOTATION_MAX:.2f}–{FIVE_STATE_PROBABLE_ANNOTATION_MAX:.2f}',COLOUR_PROBABLY_ANNOTATION),
        (f'BLUE {FIVE_STATE_PROBABLE_ANNOTATION_MAX:.2f}–{FIVE_STATE_UNSURE_MAX:.2f}',COLOUR_UNSURE),
        (f'BLACK p >= {FIVE_STATE_UNSURE_MAX:.2f}',(0,0,0))])

    truth_masks=[]
    for truth,bgr in TRUTH_BGR.items():
        m=np.zeros((h,w),bool); hit=(g.truth==truth).to_numpy() & (g.p_print>=REVIEW_THRESHOLD).to_numpy(); m[ys[hit],xs[hit]]=True; truth_masks.append((dilate(m),bgr))
    truth_overlay=legend(blend_masks(observed,truth_masks),f'{page} — POST-HOC truth for p >= {REVIEW_THRESHOLD:.2f}',[('PRINT truth',TRUTH_BGR[1]),('ANNOTATION truth',TRUTH_BGR[0]),('UNCERTAIN truth',TRUTH_BGR[2])])

    increments=[]
    incremental_defs=[
        ('new_at_090_vs_095.png','new at 0.90 vs 0.95',(g.auto_survives & ~g.p095_survives).to_numpy(),(255,255,0)),
        ('new_at_085_vs_090.png','new at 0.85 vs 0.90',(g.p085_survives & ~g.auto_survives).to_numpy(),(255,120,0)),
        ('new_at_080_vs_085.png','new at 0.80 vs 0.85',(g.review_survives & ~g.p085_survives).to_numpy(),(180,0,255)),
    ]
    for filename,label,hit,bgr in incremental_defs:
        m=np.zeros((h,w),bool); m[ys[hit],xs[hit]]=True
        increments.append((filename,legend(blend_masks(observed,[(dilate(m),bgr)]),f'{page} — {label}',[(label.replace('new at ',''),bgr)])))

    auto_mask=np.zeros((h,w),bool); hit=g.auto_survives.to_numpy(); auto_mask[ys[hit],xs[hit]]=True
    review_mask=np.zeros((h,w),bool); hit=(g.review_survives & ~g.auto_survives).to_numpy(); review_mask[ys[hit],xs[hit]]=True
    annotation_mask=np.zeros((h,w),bool); hit=((1.0-g.p_print)>=ANNOTATION_THRESHOLD).to_numpy(); annotation_mask[ys[hit],xs[hit]]=True
    tier=legend(blend_masks(observed,[(dilate(auto_mask),(0,220,0)),(dilate(review_mask),(0,191,255)),(dilate(annotation_mask),(0,0,255))]),f'{page} — V8.2b tiered diagnostic',[('AUTO >=0.90',(0,220,0)),('REVIEW 0.80–0.90',(0,191,255)),('ANNOTATION >=0.995',(0,0,255))])

    candidate=v78.copy(); candidate[auto_mask]=observed[auto_mask]
    auth=page_context[page]['mask_bool']; outside=int(np.sum(candidate[~auth]!=observed[~auth])); nonprint=int(np.sum(candidate[~auto_mask]!=v78[~auto_mask])); wrong=int(np.sum((candidate!=v78)&(candidate!=observed)))
    if outside or nonprint or wrong: raise RuntimeError(f'{page}: candidate integrity failure')

    d=WHOLE/page; d.mkdir(parents=True,exist_ok=False)
    images=[
        ('observed.png',observed),('v7_8.png',v78),('v8_2_original.png',v82),('v8_2b_auto090.png',candidate),
        ('probability_heatmap.png',heat),('threshold_band_overlay.png',band_overlay),('five_state_confidence_overlay.png',five),
        ('classifier_opinion_only.png',opinion),('classifier_opinion_on_page.png',opinion_page),('extreme_state_review_preview.png',extreme),
        ('truth_band_overlay_posthoc.png',truth_overlay),('tiered_review_overlay.png',tier)]
    for name,img in images:
        if not cv2.imwrite(str(d/name),img): raise RuntimeError(f'{page}: failed to write {name}')
    for name,img in increments:
        if not cv2.imwrite(str(d/name),img): raise RuntimeError(f'{page}: failed to write {name}')

    page_inventory.append({
        'page':page,'auto_pixels':int(auto_mask.sum()),'review_pixels':int(review_mask.sum()),
        'opinion_white_pixels':int(white.sum()),'opinion_grey_pixels':int(grey.sum()),'opinion_black_pixels':int(black.sum()),
        'outside_mask_drift':outside,'non_auto_v78_regression':nonprint,'changed_not_equal_observed':wrong,
        'candidate_sha256':sha256_file(d/'v8_2b_auto090.png'),'heatmap_sha256':sha256_file(d/'probability_heatmap.png'),
        'five_state_sha256':sha256_file(d/'five_state_confidence_overlay.png'),
        'opinion_only_sha256':sha256_file(d/'classifier_opinion_only.png'),
        'opinion_on_page_sha256':sha256_file(d/'classifier_opinion_on_page.png'),
        'extreme_preview_sha256':sha256_file(d/'extreme_state_review_preview.png'),
        'band_overlay_sha256':sha256_file(d/'threshold_band_overlay.png')})

pd.DataFrame(page_inventory).to_csv(AUDIT/'page_output_integrity_and_hashes.csv',index=False)
print('Page outputs:',len(page_inventory))
print('Opinion experiment:',json.dumps(black_summary,indent=2))


## 10. Component summary overlays and high-value audit queues

In [ ]:
# Rank page-band combinations for focused review.
rank=(enriched[enriched.p_print>=REVIEW_THRESHOLD].groupby(['page','source_group','band_id'],as_index=False).agg(pixels=('p_print','size'),true_print=('truth',lambda x:int((x==1).sum())),false_annotation=('truth',lambda x:int((x==0).sum())),truth_uncertain=('truth',lambda x:int((x==2).sum())),mean_probability=('p_print','mean'),max_run=('review_run_size','max')))
rank['utility']=rank.true_print-5*rank.false_annotation
rank.to_csv(TABLES/'page_band_review_queue.csv',index=False)

# Component bounding boxes on observed pages for components containing p >= 0.80.
for page,all_page_rows in enriched.groupby('page'):
    g=all_page_rows[all_page_rows.p_print>=REVIEW_THRESHOLD]
    image=cv2.imread(str(SOURCE/'whole_pages'/page/'observed.png'),cv2.IMREAD_COLOR)
    ctx=page_context[page]; labels=ctx['component_id']; ids=sorted(set(labels[g.y.to_numpy(int),g.x.to_numpy(int)].tolist())-{0})
    for cid in ids:
        ys,xs=np.where(labels==cid); x0,x1=int(xs.min()),int(xs.max()); y0,y1=int(ys.min()),int(ys.max())
        q=g[labels[g.y.to_numpy(int),g.x.to_numpy(int)]==cid]; tp=int((q.truth==1).sum()); fp=int((q.truth==0).sum()); tu=int((q.truth==2).sum())
        colour=(0,220,0) if tp>=max(fp,tu) else ((0,0,255) if fp>=tu else (0,191,255))
        cv2.rectangle(image,(x0,y0),(x1,y1),colour,2); cv2.putText(image,f'{cid}:P{tp}/A{fp}/U{tu}',(x0,max(12,y0-4)),cv2.FONT_HERSHEY_SIMPLEX,.38,colour,1,cv2.LINE_AA)
    cv2.imwrite(str(WHOLE/page/'component_summary_overlay.png'),image)
print('Component overlays written.')

## 11. Contact sheets

In [ ]:
def make_sheet(page_order,name,files,max_pages=CONTACT_SHEET_PAGES):
    pages=list(page_order)[:max_pages]
    if not pages:return
    fig,axes=plt.subplots(len(pages),len(files),figsize=(5*len(files),5.8*len(pages)),squeeze=False)
    for r,page in enumerate(pages):
        for c,(filename,title) in enumerate(files):
            im=cv2.imread(str(WHOLE/page/filename),cv2.IMREAD_UNCHANGED)
            if im is not None and im.ndim==3: im=cv2.cvtColor(im,cv2.COLOR_BGR2RGB)
            axes[r,c].imshow(im,cmap='gray' if im is not None and im.ndim==2 else None); axes[r,c].set_title(f'{page} — {title}'); axes[r,c].axis('off')
    plt.tight_layout(); plt.savefig(VISUAL/f'{name}.png',dpi=120,bbox_inches='tight'); plt.close(fig)

page_score=(enriched.groupby('page',as_index=False).agg(high=('auto_survives','sum'),review=('review_survives','sum'),mean_p=('p_print','mean'))).sort_values(['high','review'],ascending=False)
order=page_score.page.tolist()
make_sheet(order,'top_probability_band_pages',[('v7_8.png','V7.8'),('v8_2b_auto090.png','auto 0.90'),('threshold_band_overlay.png','bands')])
make_sheet(order,'top_heatmap_and_truth_pages',[('probability_heatmap.png','heatmap'),('truth_band_overlay_posthoc.png','post-hoc truth'),('component_summary_overlay.png','components')])
make_sheet(order,'five_state_and_opinion_experiment_pages',[
    ('five_state_confidence_overlay.png','five-state confidence'),
    ('classifier_opinion_only.png','opinion only'),
    ('classifier_opinion_on_page.png','opinion on page'),
    ('extreme_state_review_preview.png','extreme review preview')])
make_sheet(order,'incremental_threshold_pages',[('new_at_090_vs_095.png','0.90–0.95'),('new_at_085_vs_090.png','0.85–0.90'),('new_at_080_vs_085.png','0.80–0.85')])
print('Contact sheets:',VISUAL)

## 12. Final identity, record and summary

In [ ]:
environment={'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}
(OUT/'software_environment.json').write_text(json.dumps(environment,indent=2))
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.2b-spatial-probability-band-audit','development_only':True,'promotion_authority':False,'retrained':False,'refitted':False,'recalibrated':False,'source_v82_run':str(SOURCE),'source_record':source_record,'bands':BANDS,'five_state_definitions':FIVE_STATE_DEFINITIONS,'opinion_experiment':{'white_below':OPINION_WHITE_BELOW,'black_from':OPINION_BLACK_FROM,'grey_value':OPINION_GREY_VALUE,'connected_run_filter':OPINION_APPLY_CONNECTED_RUN_FILTER,'minimum_black_run':OPINION_MIN_BLACK_RUN,'diagnostic_only':True},'strict_threshold':STRICT_THRESHOLD,'auto_threshold':AUTO_THRESHOLD,'review_threshold':REVIEW_THRESHOLD,'annotation_threshold':ANNOTATION_THRESHOLD,'run_filter':ENABLE_RUN_FILTER,'minimum_print_run':MIN_PRINT_RUN,'band_predictions_sha256':sha256_file(TABLES/'band_enriched_oof_predictions.parquet'),'page_integrity_sha256':sha256_file(AUDIT/'page_output_integrity_and_hashes.csv'),'software':environment}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))
summary=pd.read_csv(TABLES/'diagnostic_policy_summary.csv')
lines=['V8.2b SPATIAL PROBABILITY-BAND AUDIT','',f'Source V8.2 run: {SOURCE}',f'V8.2b run: {OUT}','Retrained: False','Refitted: False','Recalibrated: False','',summary.to_string(index=False),'','Interpretation: development localisation and review evidence only. No threshold or policy is frozen or authorised for independent or production use.','The complete V8.2 source record, models and calibrators were copied and hash-verified inside this V8.2b run.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines)); print('\nV8.2b COMPLETE:',OUT)

## Outputs to inspect first

- `executive_summary.txt`
- `tables/band_summary_pooled.csv`
- `tables/band_summary_by_page.csv`
- `tables/band_summary_by_source_group.csv`
- `tables/route_context_band_analysis.csv`
- `tables/support_band_analysis.csv`
- `tables/component_band_analysis.csv`
- `tables/incremental_threshold_by_page.csv`
- `tables/opinion_experiment_truth_summary_pooled.csv`
- `tables/opinion_experiment_truth_summary_by_page.csv`
- `tables/opinion_experiment_truth_summary_by_source_group.csv`
- `tables/opinion_experiment_summary.json`
- `visual_audit/top_probability_band_pages.png`
- `visual_audit/top_heatmap_and_truth_pages.png`
- `visual_audit/five_state_and_opinion_experiment_pages.png`
- `visual_audit/incremental_threshold_pages.png`
- `whole_pages/<page>/probability_heatmap.png`
- `whole_pages/<page>/five_state_confidence_overlay.png`
- `whole_pages/<page>/classifier_opinion_only.png` — white/grey/black synthetic opinion canvas
- `whole_pages/<page>/classifier_opinion_on_page.png` — same opinion states on observed-page context
- `whole_pages/<page>/extreme_state_review_preview.png` — definite annotation white, print-facing bands black, orange/blue retained
- `whole_pages/<page>/threshold_band_overlay.png`
- `whole_pages/<page>/tiered_review_overlay.png`
- `whole_pages/<page>/truth_band_overlay_posthoc.png`
- `whole_pages/<page>/component_summary_overlay.png`
- `audit/copied_source_v82_artifact_inventory.csv`
- `audit/page_output_integrity_and_hashes.csv`
- `source_v82_record.json`

The `truth_band_overlay_posthoc.png` and truth columns are development diagnostics only. They must not become inference inputs.

The classifier-opinion images are synthetic development diagnostics only. They are not corrected pages and do not modify the registered V8.2b automatic candidate, thresholds, calibration, V7.8 fallback or output-integrity contract.